# Colebrookova jednadžba: predvidi → izračunaj → provjeri

Moodyjev dijagram nije samo slika: u turbulentnom području svaka njegova krivulja zadovoljava implicitnu Colebrookovu jednadžbu za Darcyjev faktor trenja \(\lambda\).

## Predvidi

Prije iteracije procijeni:

1. Povećava li relativna hrapavost \(\varepsilon/D\) faktor trenja?
2. Hoće li dvije razumne početne pretpostavke završiti na istom korijenu?
3. Zašto Colebrookovu jednadžbu ne treba primjenjivati na laminarni tok?

Prijelazno područje \(2300\lesssim Re\lesssim4000\) ovdje namjerno ne interpoliramo: režim može biti nestabilan i jedna glatka formula skriva tu neizvjesnost.

Nastavak povezuje Colebrookov račun s radnom točkom iz **Z4 poglavlja 13**. Predvidi kako će veća hrapavost pomaknuti presjek crpke i sustava. Sintetički podatci služe nastavi, nisu krivulja određene komercijalne crpke.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

def colebrook_residual(lam, Re, rel_roughness):
    return 1/np.sqrt(lam) + 2*np.log10(rel_roughness/3.7 + 2.51/(Re*np.sqrt(lam)))

def colebrook_iter(Re, rel_roughness, x0=7.0, tol=1e-12, max_iter=100):
    # Fiksna točka u varijabli x=1/sqrt(lambda), uz povijest iteracija.
    if Re <= 4000 or rel_roughness < 0:
        raise ValueError("Colebrookov račun ovdje vrijedi za Re > 4000 i ε/D ≥ 0.")
    x = float(x0)
    history = []
    for iteration in range(max_iter + 1):
        lam = 1/x**2
        residual = colebrook_residual(lam, Re, rel_roughness)
        history.append((iteration, lam, residual))
        x_new = -2*np.log10(rel_roughness/3.7 + 2.51*x/Re)
        if abs(x_new-x) < tol:
            x = x_new
            lam = 1/x**2
            history.append((iteration+1, lam, colebrook_residual(lam, Re, rel_roughness)))
            return lam, np.asarray(history)
        x = x_new
    raise RuntimeError("Iteracija nije konvergirala unutar zadanog broja koraka.")

Re0, rr0 = 1.0e5, 1.0e-4
lam0, history = colebrook_iter(Re0, rr0, x0=5.0)
print(f"lambda = {lam0:.8f}; iteracija = {len(history)-1}; završni rezidual = {history[-1,2]:.3e}")
print(" i       lambda       rezidual")
for row in history:
    print(f"{int(row[0]):2d}   {row[1]:.9f}   {row[2]: .3e}")


## Izračunaj: konvergencija i osjetljivost na hrapavost

Rezidual je lijeva strana implicitne jednadžbe i mora težiti nuli. Zatim za isti Reynoldsov broj mijenjamo \(\varepsilon/D\) kroz četiri reda veličine. To je numerička verzija horizontalnog presjeka Moodyjeva dijagrama.


In [ ]:
lam_other_start, history_other = colebrook_iter(Re0, rr0, x0=12.0)
roughness = np.logspace(-6, -2, 70)
lam_rough = np.array([colebrook_iter(Re0, rr)[0] for rr in roughness])

def friction_factor(Re, rel_roughness=0.0):
    if Re < 2300:
        return 64/Re
    if Re <= 4000:
        raise ValueError("Prijelazno područje nema jedinstvenu vrijednost u ovom modelu.")
    return colebrook_iter(Re, rel_roughness)[0]

lam_laminar = friction_factor(1200, rr0)
print(f"Laminarno, Re=1200: lambda = {lam_laminar:.6f}")
print(f"Turbulentno: promjena λ od {lam_rough[0]:.5f} do {lam_rough[-1]:.5f}")


## Provjeri

Neovisne provjere su: zatvaranje implicitne jednadžbe, neovisnost korijena o početnoj pretpostavci i analitički laminarni granični slučaj \(\lambda=64/Re\).


In [ ]:
assert abs(colebrook_residual(lam0, Re0, rr0)) < 1e-10
assert np.isclose(lam_other_start, lam0, rtol=1e-11)
assert np.isclose(lam_laminar, 64/1200, rtol=1e-14)
assert np.all(np.diff(lam_rough) > 0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].semilogy(history[:,0], np.maximum(np.abs(history[:,2]), 1e-16), "o-", color="#b43c35")
axes[0].set(xlabel="iteracija", ylabel="|Colebrookov rezidual|", title="Povijest konvergencije")
axes[1].semilogx(roughness, lam_rough, color="#256d85", lw=2)
axes[1].scatter([rr0], [lam0], color="#b43c35", zorder=3)
axes[1].set(xlabel=r"relativna hrapavost $\varepsilon/D$", ylabel=r"Darcyjev $\lambda$", title=f"Osjetljivost pri Re={Re0:.0e}")
for ax in axes: ax.grid(True, which="both", ls=":", alpha=.45)
plt.tight_layout(); plt.show()


## Z4 poglavlja 13: trenje i radna točka u istoj iteraciji

Crpka ima $H_p=30-30000Q^2$, uz $Q$ u m³/s i visinu u metrima. Između velikih otvorenih spremnika je $Δz=8$ m. Vod ima $D=0{,}100$ m, $L=150$ m, $ε=0{,}100$ mm i $Σξ=6$, uključujući ulaz i izlaz. Koeficijenti se odnose na brzinu u toj cijevi. Za vodu vrijedi $ρ=1000$ kg/m³, $ν=10^{-6}$ m²/s; ukupna učinkovitost crpke je 0,76, motora 0,92. Gubitci pretvarača su zanemareni.

Za svaku probnu vrijednost protoka ponovno izračunavamo $Re$ i rješavamo Colebrookovu jednadžbu. Vanjska bisekcija zatvara $H_p-H_s=0$. Provjeravamo oba reziduala i konačni turbulentni režim; samo mali korak iteracije nije dovoljan.

In [ ]:
D4, L4, eps4, xi4 = 0.100, 150.0, 0.100e-3, 6.0
nu4, rho4, g4, dz4 = 1e-6, 1000.0, 9.81, 8.0
eta_p4, eta_m4 = 0.76, 0.92
def pump_state4(Q, epsilon=eps4):
    v = 4*Q/(np.pi*D4**2)
    Re = v*D4/nu4
    lam, _ = colebrook_iter(Re, epsilon/D4)
    Hp = 30-30000*Q**2
    Hs = dz4+(lam*L4/D4+xi4)*v**2/(2*g4)
    return Hp, Hs, Re, lam

def operating_point4(epsilon=eps4, tolerance=1e-10):
    lo, hi = 0.005, 0.030
    alo, blo, *_ = pump_state4(lo, epsilon)
    ahi, bhi, *_ = pump_state4(hi, epsilon)
    if not alo>blo or not ahi<bhi:
        raise ValueError("Interval ne obuhvaća radnu točku.")
    history = []
    for iteration in range(80):
        Q = (lo+hi)/2
        Hp, Hs, Re, lam = pump_state4(Q, epsilon)
        residual = Hp-Hs
        history.append((iteration,Q,residual))
        if abs(residual)<tolerance:
            return Q, Hp, Re, lam, np.array(history)
        if residual>0:lo=Q
        else:hi=Q
    raise RuntimeError("Radna točka nije konvergirala.")

Q4, H4, Re4, lam4, history4 = operating_point4()
Ph4 = rho4*g4*Q4*H4
Pvr4 = Ph4/eta_p4
Pel4 = Pvr4/eta_m4
print(f"Q = {Q4*1000:.6f} L/s; H = {H4:.6f} m; Re = {Re4:.3f}; lambda = {lam4:.9f}")
print(f"Ph = {Ph4/1000:.6f} kW; Pvr = {Pvr4/1000:.6f} kW; Pel = {Pel4/1000:.6f} kW")
print(f"Energetski rezidual = {history4[-1,2]:.3e} m; Colebrookov = {colebrook_residual(lam4, Re4, eps4/D4):.3e}")
assert abs(Q4*1000-19.030)<0.0005
assert abs(H4-19.136)<0.0005
assert abs(history4[-1,2])<1e-6
assert abs(colebrook_residual(lam4, Re4, eps4/D4))<1e-6
assert Re4>4000
assert Ph4<Pvr4<Pel4
assert np.isclose(Pel4*eta_m4*eta_p4, Ph4, rtol=1e-14)
Qrough4, Hrough4, *_ = operating_point4(epsilon=2*eps4)
assert Qrough4<Q4 and Hrough4>H4

## Provjeri krivulje i njihov pomak

Na prvom grafu obje su osi kvantitativne. Drugi prikazuje povijest stvarnog energijskog reziduala. Faktor trenja nije unaprijed zamrznut na njegovoj konačnoj vrijednosti. Statička visina ostaje 8 m.

In [ ]:
q_plot4 = np.linspace(0.005,0.030,120)
states4 = np.array([pump_state4(q) for q in q_plot4])
rough_states4 = np.array([pump_state4(q,2*eps4) for q in q_plot4])
fig, axes = plt.subplots(1,2,figsize=(10,4))
axes[0].plot(q_plot4*1000,states4[:,0],label="crpka",color="#1565c0")
axes[0].plot(q_plot4*1000,states4[:,1],label="sustav, ε = 0,100 mm",color="#c0392b")
axes[0].plot(q_plot4*1000,rough_states4[:,1],"--",label="sustav, dvostruka ε",color="#8e44ad")
axes[0].plot(Q4*1000,H4,"o",color="#3a4a56")
axes[0].set(xlabel="Q (L/s)",ylabel="H (m)",title="Z4: promjenjivo trenje i radna točka")
axes[0].legend(fontsize=8)
axes[1].semilogy(history4[:,0],np.maximum(np.abs(history4[:,2]),1e-16),"o-",color="#1e8449")
axes[1].set(xlabel="vanjska iteracija",ylabel="|Hp − Hs| (m)",title="Provjera energijske bilance")
for ax in axes:ax.grid(ls=":",alpha=.4)
plt.tight_layout();plt.show()

## Protumači

1. Kako promjena $Re$ za ±10 % utječe na faktor trenja u početnom Colebrookovu pokusu? Usporedi s udvostručenjem relativne hrapavosti.
2. Zašto u Z4 treba provjeriti energijski i Colebrookov rezidual? Može li jedan biti malen dok drugi nije?
3. Zašto se pri povećanju hrapavosti radni protok smanjuje, a visina na zadanoj crpkinoj krivulji povećava?
4. Što bi bilo pogrešno u računu radne točke s unaprijed fiksnim $λ=0{,}02$? Koliko se promijeni rezultat, a koliko model?
5. Zašto najveća od triju snaga pripada električnom ulazu? Koji gubitci nisu sadržani u hidrauličkoj snazi?
6. Može li se zadana sintetička krivulja koristiti izvan zadanog raspona protoka bez dodatnih podataka?